# Privileged debrief (mode 4)

Interactively analyze any **recorded** play conversation with full access to
ground truth: the saved snapshot images *and* the exact Settings JSON
(coordinates, angles) that the playing agent never saw.

Things to know:

- **The playing agent never sees this conversation.** Debriefs are stored as
  their own NAMS conversations (`kind='debrief'`, linked to the play
  conversation via a `DEBRIEF_OF` edge). The only way debrief content reaches
  the play conversation is the explicit **Finalize & save self-eval** button,
  which distills the debrief into a mode-3-style verdict
  (`kind='self_evaluation'` message + reasoning trace).
- **Switching the play conversation starts a fresh debrief conversation.**
  The previous one stays stored in NAMS; this panel just stops appending to it.
- **The model navigates the player's recorded messages** (numbered 0..M:
  moves, reflections, and answers) with `[SHOW <n>]`, `[NEXT]`, and `[BACK]`.
  Its context always holds exactly ONE "current message": the recorded text,
  the single frame the player saw at that moment, and that frame's settings —
  plus the user instruction the player was answering, kept permanently in
  view. The cursor starts at message 0. Multi-generation replies are normal —
  you'll see each generation and the fetched frame as they happen.
- **The model can also search on its own** with `[SEARCH <query>]`: semantic
  search over this play session's recorded messages (hits come back labeled
  with their `[SHOW n]` numbers) plus the general memory tiers (tips, past
  reasoning). Searches share the same per-turn tool budget as navigation.
- **Tips.** The analyst can persist a one-line lesson to long-term memory
  (`[WRITE_TIP]` + a `TIP: ...` line), making it retrievable in every future
  mode including live play. The prompt requires your explicit approval of the
  exact wording first — it may *suggest* a tip, but you decide. Ask it to
  save one (e.g. "yes, save exactly that wording") when you agree.
- **Model dropdown.** The picker at the top of the chat panel switches
  between the registry models (see `MODEL_CANDIDATES.md`). With **"Save only
  one set of weights at a time"** checked, a switch starts a fresh debrief
  thread (over the same play conversation) and deletes the other registry
  models' cached weights from disk.
- **GPU caveat:** this notebook loads its own model copy (Gemma 4 E4B ~16 GB
  by default). If the play notebook is open with its model loaded, close it
  (or run on a box with VRAM for two).

Prereqs: Neo4j is up, and there is at least one recorded play conversation.

In [1]:
import os
import sys

# Run from the repo root so `agent` imports and `.env` resolve.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from agent.debrief import DebriefSession

# Connects NAMS on a background loop and loads Gemma 4 E4B (first run is slow).
# Logging is on by default: every LLM call and DB retrieval lands under
# session.logger.run_dir.
session = DebriefSession()
if session.logger is not None:
    print("log dir:", session.logger.run_dir)

pygame 2.6.1 (SDL 2.28.4, Python 3.12.13)
Hello from the pygame community. https://www.pygame.org/contribute.html


/venv/main/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

log dir: logs/debrief_2026-07-24_18-49-41


In [2]:
import ipywidgets as widgets
from IPython.display import Image, display

from agent.notebook_ui import model_picker, tame_shift_enter

FRAME_WIDTH = 420  # px; frames render at cfg.game_size but display fixed-width

picker_out = widgets.Output()


def _conversation_options():
    """(label, session_id) pairs for the dropdown -- play conversations only."""
    options = []
    for c in session.list_conversations():
        sid = c["session_id"]
        label = f"{sid[:8]}… — {c.get('created_at')}, {c.get('n_messages')} msgs"
        options.append((label, sid))
    return options


dropdown = widgets.Dropdown(
    options=_conversation_options(),
    value=None,
    description="Play conv:",
    layout=widgets.Layout(width="600px"),
)
refresh_btn = widgets.Button(description="Refresh list", button_style="info")


def _on_select(change):
    if change["new"] is None:
        return
    with picker_out:
        picker_out.clear_output()
        info = session.select(change["new"])
        print(f"Analyzing play conversation: {info['play_session_id']}")
        print(f"New debrief conversation:    {info['debrief_session_id']}")
        print(
            f"({info['n_player_messages']} player messages, "
            f"{info['n_moves']} moves, {info['n_snapshots']} snapshots)"
        )
        if info["latest_frame_path"]:
            print("-- latest saved frame (current state) --")
            display(Image(filename=info["latest_frame_path"], width=FRAME_WIDTH))


def _on_refresh(_):
    # Sessions recorded after notebook start appear here; the current
    # selection (and its debrief conversation) is untouched.
    dropdown.unobserve(_on_select, names="value")
    selected = dropdown.value
    dropdown.options = _conversation_options()
    dropdown.value = selected if selected in [v for _, v in dropdown.options] else None
    dropdown.observe(_on_select, names="value")


dropdown.observe(_on_select, names="value")
refresh_btn.on_click(_on_refresh)

display(widgets.VBox([widgets.HBox([dropdown, refresh_btn]), picker_out]))

In [3]:
question_box = widgets.Textarea(
    value="",
    placeholder="Ask about the recorded session (e.g. 'Was the agent ever facing the gold?')...",
    description="You:",
    layout=widgets.Layout(width="600px", height="70px"),
)
ask_btn = widgets.Button(description="Ask", button_style="primary")
save_btn = widgets.Button(
    description="Finalize & save self-eval",
    button_style="warning",
    layout=widgets.Layout(width="200px"),
)
dump_btn = widgets.Button(description="Dump DB status", button_style="info")
chat_out = widgets.Output()


def _show_generation(step):
    """Render one generation of a debrief turn as it happens (wired to
    ``session.ask(on_step=...)``)."""
    print(f"Analyst: {step['text']}")
    if step.get("missing_think_close"):
        print(
            "[FORMAT ERROR: thinking model never closed its think block -- "
            "full raw text kept visible]"
        )
    call = step["tool_call"]
    if call is None:
        return
    tool = call["tool"]
    if tool == "SEARCH":
        print(f"[tool call: SEARCH {call['query']!r} -- results fed back]")
    elif tool == "WRITE_TIP":
        if step.get("tip_saved"):
            print(f"[tool call: WRITE_TIP -> saved as {step['tip_saved']['category']}]")
        else:
            print("[tool call: WRITE_TIP without a 'TIP:' line -- nothing saved]")
    else:
        desc = " ".join(str(v) for v in call.values())  # e.g. 'SHOW 42', 'NEXT'
        print(f"[tool call: {desc} -> current message: {step['cursor']}]")
        for path in step["frames"]:
            display(Image(filename=path, width=FRAME_WIDTH))
        if not step["frames"]:
            print("(invalid call -- the model was told why)")


def _on_ask(_):
    q = question_box.value.strip()
    if not q:
        return
    ask_btn.disabled = True
    try:
        with chat_out:
            print(f"\n=== You: {q}")
            result = session.ask(q, on_step=_show_generation)
            print(
                f"[turn ended: {result['num_generations']} generation(s), "
                f"{result['tool_calls']} tool call(s)]"
            )
        question_box.value = ""
    finally:
        ask_btn.disabled = False


def _on_save(_):
    save_btn.disabled = True
    try:
        with chat_out:
            info = session.save_self_eval()
            print("\n*** Self-eval verdict saved to the play conversation ***")
            print(f"message id: {info['eval_message_id']}")
            print(f"trace id:   {info['trace_id']}")
            print(f"--- verdict ---\n{info['verdict']}")
    finally:
        save_btn.disabled = False


def _on_dump(_):
    """Snapshot the whole memory graph to a .dump JSON file in this run's log
    directory. Reads over the live connection -- safe at any point, does NOT
    stop Neo4j. Embeddings are dropped (huge, not human-useful). Logical
    snapshot for inspection, distinct from the native binary `neo4j-admin`
    dump produced by `scripts/neo4j_db.sh save`."""
    dump_btn.disabled = True
    try:
        with chat_out:
            info = session.dump_db()
            print(f"\nDB dumped -> {info['path']}")
            print(f"  nodes = {info['nodes']}   relationships = {info['relationships']}")
            if session.logger is not None:
                print(f"  llm log: {session.logger.llm_txt}")
                print(f"  db  log: {session.logger.db_txt}")
    finally:
        dump_btn.disabled = False


ask_btn.on_click(_on_ask)
save_btn.on_click(_on_save)
dump_btn.on_click(_on_dump)


def _on_model_switched(info):
    """Refresh after a model switch. With the one-weights checkbox the
    debrief thread was restarted (a fresh conversation over the same play
    session, if one was selected); without it the thread continues."""
    with chat_out:
        if info["restarted"]:
            restart = info.get("restart") or {}
            if restart.get("debrief_session_id"):
                print(f"\n[model switched to {info['label']} -- fresh debrief "
                      f"thread {restart['debrief_session_id']} over the same "
                      f"play conversation]")
            else:
                print(f"\n[model switched to {info['label']} -- select a play "
                      f"conversation above to start a debrief]")
        else:
            print(f"\n[model switched to {info['label']} -- debrief thread continues]")


# Model picker on top, then conversation, then input: the box stays next to
# the newest output, no scrolling back up after a long exchange.
display(widgets.VBox([
    model_picker(session, on_switched=_on_model_switched),
    chat_out,
    question_box,
    widgets.HBox([ask_btn, save_btn, dump_btn]),
]))
# Shift+Enter = plain newline in the box (never submits, never re-runs the cell).
tame_shift_enter(question_box)

When you're done, release the model and close the memory client:

In [ ]:
session.close()